# 🖼️ Image Captioning with CNN + LSTM
**Encoder-Decoder Architecture | ResNet50 + LSTM | GloVe Embeddings | BLEU/METEOR**

This notebook walks through every stage of the project:
1. Setup & Data Exploration
2. Vocabulary Building with GloVe
3. Dataset & DataLoaders
4. Model Architecture
5. Training
6. Inference (Greedy + Beam Search)
7. BLEU & METEOR Evaluation
8. Qualitative Analysis

---
## 0. Setup

In [ ]:
import os, sys
# Make sure the project root is on the path
PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
sys.path.insert(0, PROJECT_ROOT)

import torch
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display
from PIL import Image
import numpy as np

import config

print(f'PyTorch version : {torch.__version__}')
print(f'Device          : {config.DEVICE}')
print(f'CUDA available  : {torch.cuda.is_available()}')

---
## 1. Data Exploration
### 1.1 Peek at captions

In [ ]:
from utils.dataset import parse_captions

image_captions = parse_captions()
print(f'Total images: {len(image_captions)}')

# Show a random image with its 5 captions
sample_img = list(image_captions.keys())[42]
print(f'\nSample image: {sample_img}')
for i, cap in enumerate(image_captions[sample_img], 1):
    print(f'  Caption {i}: {cap}')

### 1.2 Visualise sample images

In [ ]:
import random

sample_keys = random.sample(list(image_captions.keys()), 6)

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
for ax, img_name in zip(axes.flatten(), sample_keys):
    img_path = os.path.join(config.IMAGES_DIR, img_name)
    if os.path.exists(img_path):
        img = Image.open(img_path)
        ax.imshow(img)
        caption = image_captions[img_name][0]  # first caption
        ax.set_title(caption[:60] + '…' if len(caption) > 60 else caption,
                     fontsize=9, wrap=True)
    ax.axis('off')

plt.suptitle('Flickr8k Sample Images with Captions', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### 1.3 Caption length distribution

In [ ]:
from utils.vocabulary import tokenize

all_caps   = [c for caps in image_captions.values() for c in caps]
lengths    = [len(tokenize(c)) for c in all_caps]

plt.figure(figsize=(9, 4))
plt.hist(lengths, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
plt.axvline(np.mean(lengths), color='red', linestyle='--', label=f'Mean = {np.mean(lengths):.1f}')
plt.axvline(config.MAX_CAPTION_LENGTH, color='orange', linestyle='--',
            label=f'Max length = {config.MAX_CAPTION_LENGTH}')
plt.xlabel('Caption length (tokens)')
plt.ylabel('Count')
plt.title('Caption Length Distribution')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Total captions : {len(lengths):,}')
print(f'Mean length    : {np.mean(lengths):.1f} tokens')
print(f'Max length     : {max(lengths)} tokens')
print(f'% within {config.MAX_CAPTION_LENGTH} tokens: {100*np.mean([l <= config.MAX_CAPTION_LENGTH for l in lengths]):.1f}%')

---
## 2. Vocabulary & GloVe

In [ ]:
from utils.vocabulary import Vocabulary
import os

# Build or load vocabulary
if os.path.exists(config.VOCAB_PATH):
    vocab = Vocabulary.load()
else:
    vocab = Vocabulary()
    vocab.build_from_captions(all_caps, min_freq=config.MIN_WORD_FREQ)
    vocab.save()

print(f'\nVocab size: {len(vocab):,}')

# Show most common words
top20 = vocab.word_freq.most_common(20)
words, counts = zip(*top20)
plt.figure(figsize=(10, 4))
plt.bar(words, counts, color='teal', alpha=0.8)
plt.title('Top 20 Most Frequent Words')
plt.xticks(rotation=45, ha='right')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

In [ ]:
# Load GloVe (if available)
glove_matrix = None
if config.USE_GLOVE and os.path.exists(config.GLOVE_FILE):
    glove_matrix = vocab.build_glove_matrix()
    print(f'GloVe matrix shape: {glove_matrix.shape}')
else:
    print('GloVe not available — embeddings will be randomly initialised.')
    print(f'Expected file: {config.GLOVE_FILE}')

---
## 3. Dataset & DataLoaders

In [ ]:
from utils.dataset import build_dataloaders

train_loader, val_loader, test_loader, test_refs = build_dataloaders(
    vocab, batch_size=config.BATCH_SIZE
)

# Inspect one batch
images, captions, lengths = next(iter(train_loader))
print(f'images   shape : {images.shape}    dtype: {images.dtype}')
print(f'captions shape : {captions.shape}  dtype: {captions.dtype}')
print(f'lengths  shape : {lengths.shape}   dtype: {lengths.dtype}')
print(f'\nSample caption (encoded): {captions[0].tolist()}')
print(f'Sample caption (decoded): {vocab.decode(captions[0].tolist())}')

---
## 4. Model Architecture

In [ ]:
from model import ImageCaptioningModel

model = ImageCaptioningModel(
    vocab_size    = len(vocab),
    embed_dim     = config.EMBED_DIM,
    hidden_dim    = config.HIDDEN_DIM,
    num_layers    = config.NUM_LAYERS,
    dropout       = config.DROPOUT,
    glove_matrix  = glove_matrix,
    pad_idx       = vocab[config.PAD_TOKEN],
).to(config.DEVICE)

total_params   = sum(p.numel() for p in model.parameters())
trainable      = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen         = total_params - trainable

print(f'Model Architecture')
print(f'  Total params     : {total_params:>12,}')
print(f'  Trainable params : {trainable:>12,}')
print(f'  Frozen params    : {frozen:>12,}  (CNN backbone)')

# Test forward pass
dummy_imgs = torch.randn(4, 3, 224, 224).to(config.DEVICE)
dummy_caps = torch.randint(0, len(vocab), (4, config.MAX_CAPTION_LENGTH)).to(config.DEVICE)
with torch.no_grad():
    logits = model(dummy_imgs, dummy_caps)
print(f'\nForward pass OK: images {list(dummy_imgs.shape)} → logits {list(logits.shape)}')

---
## 5. Training
> **Tip**: Run `python train.py` from the terminal for the full training loop with progress bars. Here we train for 1 mini-epoch to verify everything works.

In [ ]:
import torch.nn as nn
import torch.optim as optim

pad_idx   = vocab[config.PAD_TOKEN]
criterion = nn.CrossEntropyLoss(ignore_index=pad_idx, label_smoothing=0.1)
optimizer = optim.Adam(model.trainable_parameters(), lr=config.LEARNING_RATE)

# One mini-epoch (just a few batches)
model.train()
mini_losses = []
N_MINI_BATCHES = 10

for i, (imgs, caps, lens) in enumerate(train_loader):
    if i >= N_MINI_BATCHES:
        break

    imgs = imgs.to(config.DEVICE)
    caps = caps.to(config.DEVICE)

    logits  = model(imgs, caps)               # (B, T-1, V)
    targets = caps[:, 1:].contiguous()
    B, T, V = logits.shape
    loss    = criterion(logits.view(B*T, V), targets.view(B*T))

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), config.CLIP_GRAD_NORM)
    optimizer.step()

    mini_losses.append(loss.item())
    print(f'  Batch {i+1:2d}/{N_MINI_BATCHES}  loss={loss.item():.4f}')

plt.figure(figsize=(7, 3))
plt.plot(mini_losses, 'b-o', markersize=5)
plt.xlabel('Batch')
plt.ylabel('Loss')
plt.title('Mini-training Loss (verification run)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 6. Load Trained Model & Generate Captions

In [ ]:
from inference import load_model, load_image, greedy_decode, beam_search_decode

if os.path.exists(config.BEST_MODEL_PATH):
    trained_model = load_model(config.BEST_MODEL_PATH, vocab, config.DEVICE)
    print('Trained model loaded.')
else:
    print('No trained model found. Run: python train.py')
    print('Using randomly initialised model for demo...')
    trained_model = model
    trained_model.eval()

In [ ]:
# Caption a few test images
test_images = random.sample(list(test_refs.keys()), 4)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, img_name in zip(axes.flatten(), test_images):
    img_path = os.path.join(config.IMAGES_DIR, img_name)
    if not os.path.exists(img_path):
        ax.axis('off')
        continue

    img_tensor = load_image(img_path, config.DEVICE)

    # Greedy
    greedy_cap = greedy_decode(trained_model, img_tensor, vocab)

    # Beam search top-1
    beam_results = beam_search_decode(trained_model, img_tensor, vocab, beam_size=5)
    beam_cap = beam_results[0][0]

    img = Image.open(img_path)
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(
        f'Greedy : {greedy_cap}\nBeam-5 : {beam_cap}',
        fontsize=8.5, loc='left'
    )

plt.suptitle('Generated Captions (Greedy vs Beam Search)', fontsize=13)
plt.tight_layout()
plt.show()

---
## 7. BLEU & METEOR Evaluation

In [ ]:
from utils.metrics import corpus_bleu, corpus_meteor, print_scores
from tqdm.notebook import tqdm

# Evaluate on a subset of test images (set to None for all)
EVAL_LIMIT = 200

eval_imgs = list(test_refs.keys())[:EVAL_LIMIT]
hypotheses, references_list = [], []

trained_model.eval()
for img_name in tqdm(eval_imgs, desc='Evaluating'):
    img_path = os.path.join(config.IMAGES_DIR, img_name)
    refs     = test_refs.get(img_name, [])
    if not os.path.exists(img_path) or not refs:
        continue

    img_tensor = load_image(img_path, config.DEVICE)
    caption    = greedy_decode(trained_model, img_tensor, vocab)

    hypotheses.append(caption)
    references_list.append(refs)

bleu   = corpus_bleu(hypotheses, references_list)
meteor = corpus_meteor(hypotheses, references_list)
print_scores(bleu, meteor)

---
## 8. Qualitative Analysis

In [ ]:
from utils.metrics import sentence_bleu

# Show good and bad examples
scored = []
for hyp, refs in zip(hypotheses, references_list):
    s = sentence_bleu(hyp, refs)
    scored.append((s['bleu4_cumulative'], hyp, refs))

scored.sort(key=lambda x: x[0])

print('=== Best Captions ===')
for score, hyp, refs in scored[-3:]:
    print(f'  BLEU-4 = {score:.3f}')
    print(f'  Generated : {hyp}')
    print(f'  Reference : {refs[0]}')
    print()

print('=== Worst Captions ===')
for score, hyp, refs in scored[:3]:
    print(f'  BLEU-4 = {score:.3f}')
    print(f'  Generated : {hyp}')
    print(f'  Reference : {refs[0]}')
    print()

---
## 9. Caption Your Own Image

In [ ]:
MY_IMAGE = 'path/to/your/image.jpg'   # ← change this!

if os.path.exists(MY_IMAGE):
    img_tensor = load_image(MY_IMAGE, config.DEVICE)

    greedy_cap = greedy_decode(trained_model, img_tensor, vocab)
    beam_results = beam_search_decode(trained_model, img_tensor, vocab, beam_size=5)

    print(f'Greedy decoding : {greedy_cap}')
    print('\nBeam search top-5:')
    for i, (cap, score) in enumerate(beam_results, 1):
        print(f'  {i}. [{score:.3f}] {cap}')

    img = Image.open(MY_IMAGE)
    plt.figure(figsize=(8, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title(beam_results[0][0], fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print(f'File not found: {MY_IMAGE}')